# 08 — Suivi des évaluations avec MLflow

Une modification des documents, du moteur de recherche ou des seuils peut changer les réponses. Ce notebook conserve les résultats de chaque évaluation pour comparer les versions du système.

**Objectif :** enregistrer les paramètres et les métriques dans MLflow, puis examiner les expériences locales.

Le parcours initialise le suivi, évalue les trois méthodes et retrouve les résultats enregistrés. Les textes des questions, des documents et des réponses ne sont pas enregistrés dans MLflow. Les mesures restent exploratoires : les six questions ne constituent pas un test indépendant.

## 1. Initialiser le suivi et les méthodes de recherche

Le stockage et le nom de l’expérience viennent de `configs/monitoring.yaml`. Chaque appel à `tracker.evaluate()` crée une exécution distincte. SQLite conserve les métadonnées ; les configurations et métriques agrégées sont sauvegardées comme artefacts locaux.

In [ ]:
from IPython.display import Markdown, display

from contextiq.ingestion.loader import DataLoader
from contextiq.monitoring.mlflow_tracking import MLflowTracker
from contextiq.retrieval.bm25 import BM25Retriever
from contextiq.retrieval.embeddings import EmbeddingRetriever
from contextiq.retrieval.tfidf import TfidfRetriever
from contextiq.settings import Settings

settings = Settings.load()
dataset = DataLoader().load()
tracker = MLflowTracker(settings)
retrievers = {
    "TF-IDF": TfidfRetriever(dataset.documents),
    "BM25": BM25Retriever(dataset.documents, **settings.retrieval.bm25.model_dump()),
    "Embeddings": EmbeddingRetriever(
        dataset.documents, **settings.retrieval.embeddings.model_dump()
    ),
}
print("Expérience :", settings.monitoring.experiment_name)

## 2. Enregistrer les résultats

Les métriques de recherche portent sur les questions avec source. Les réponses traversent l’abstention et les contrôles de sécurité avant leur évaluation. Les taux dont le dénominateur est nul ne sont pas enregistrés.

Chaque exécution conserve K, le seuil, les paramètres, les versions et les empreintes des données et du code. `evaluation_seconds` mesure la boucle d’évaluation, sans initialisation des retrievers : ce n’est pas une mesure de latence de production.

In [ ]:
tracked_runs = {}
for number, (name, retriever) in enumerate(retrievers.items(), start=1):
    run = tracker.evaluate(name, retriever, dataset)
    tracked_runs[name] = run
    print(f"\n{number}. {name}")
    print("=" * 72)
    print("Run MLflow :", run.run_id)
    print(f"Succès conjoint : {run.metrics['answer_success_rate']:.1%}")
    print(f"Taux de réponse : {run.metrics['answer_answer_rate']:.1%}")
    print(f"Durée de l’évaluation : {run.metrics['evaluation_seconds']:.3f} s")

## 3. Retrouver et comparer les exécutions

Le tableau présente les dernières évaluations terminées, y compris celles des sessions précédentes. Comparer des scores exige de vérifier les empreintes des données, K et les paramètres : un changement de jeu peut expliquer un écart.

In [ ]:
runs = tracker.client.search_runs(
    [tracker.experiment_id],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["attributes.start_time DESC"],
    max_results=20,
)
rows = [
    "| Méthode | K | Seuil | Recall@K | Succès conjoint | Run |",
    "|---|---:|---:|---:|---:|---|",
]
for run in runs:
    params, metrics = run.data.params, run.data.metrics
    recall = metrics.get("retrieval_recall")
    recall_label = f"{recall:.3f}" if recall is not None else "Non applicable"
    rows.append(
        f"| {params['method']} | {params['top_k']} | {params['threshold']} | "
        f"{recall_label} | {metrics['answer_success_rate']:.1%} | {run.info.run_id} |"
    )
display(Markdown("\n".join(rows)))

## 4. Interpréter le suivi

MLflow permet d’inspecter les paramètres, les métriques et les artefacts et de comparer les exécutions. Il ne collecte pas automatiquement les retours des conseillers et ne déclenche pas d’alertes dans cette première version.

Le [plan de suivi en production](../reports/production_monitoring.md) précise les indicateurs, les responsabilités, la revue métier et les validations après modification documentaire. Loguru reste chargé du diagnostic technique ; MLflow conserve les résultats des évaluations.